# 02 - Feature Engineering
## LoanRisk-ML

En este notebook transformamos los datos limpios del EDA en features listas para el modelo.

### Objetivos
- Crear ratio features que capturen relaciones financieras clave
- Extraer features temporales de las columnas de fecha
- Encodear variables categóricas
- Imputar valores nulos
- Construir el pipeline de transformación con ColumnTransformer


## 1. Importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder
from category_encoders import TargetEncoder
from sklearn.impute import SimpleImputer

%matplotlib inline

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


## 2. Carga de datos

In [2]:
ROOT = Path('..').resolve()
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'loan_clean.parquet')

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.2%}")
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())

Shape: (391164, 71)
Default rate: 20.15%

Tipos de datos:
float64           55
object            13
datetime64[ns]     1
int64              1
int32              1
Name: count, dtype: int64


## 3. Feature Engineering - Ratio Features

Los ratio features capturan relaciones financieras entre variables que individualmente 
tienen menos poder predictivo. Son estándar en credit scoring.

In [3]:
# Relación entre cuota mensual e ingreso anual
# Qué porcentaje del ingreso mensual representa la cuota del préstamo
df['payment_to_income'] = df['installment'] / (df['annual_inc'] / 12).replace(0, np.nan)

# Relación entre monto del préstamo e ingreso anual
# Cuántos años de salario representa el préstamo
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc'].replace(0, np.nan)

# Relación entre balance revolving y límite revolving total
# Qué tan utilizado está el crédito revolving del cliente
df['revol_util_amount'] = df['revol_bal'] / df['total_rev_hi_lim'].replace(0, np.nan)

# Relación entre deuda total y límite de crédito total
# Qué tan endeudado está el cliente en términos absolutos
df['total_debt_to_credit'] = df['total_bal_ex_mort'] / df['tot_hi_cred_lim'].replace(0, np.nan)

print("Ratio features creadas:")
print(df[['payment_to_income', 'loan_to_income', 
          'revol_util_amount', 'total_debt_to_credit']].describe())

Ratio features creadas:
       payment_to_income  loan_to_income  revol_util_amount  \
count      391110.000000   391110.000000      391018.000000   
mean            0.080014        0.220071           0.529159   
std             0.204287        0.515768           0.241436   
min             0.000123        0.000309           0.000000   
25%             0.047531        0.129438           0.348536   
50%             0.073038        0.201149           0.531292   
75%             0.105292        0.296610           0.714623   
max           123.781818      310.606061           3.332247   

       total_debt_to_credit  
count         391160.000000  
mean               0.448503  
std                0.280671  
min                0.000000  
25%                0.191137  
50%                0.426327  
75%                0.685724  
max                5.711831  


## 4. Capping de outliers en ratio features

In [4]:
ratio_features = ['payment_to_income', 'loan_to_income', 
                  'revol_util_amount', 'total_debt_to_credit']

for col in ratio_features:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)
    print(f"{col:<25} cap en percentil 99%: {cap:.4f}")

print("\nDespués del capping:")
print(df[ratio_features].describe())

payment_to_income         cap en percentil 99%: 0.1930


loan_to_income            cap en percentil 99%: 0.5000
revol_util_amount         cap en percentil 99%: 0.9883
total_debt_to_credit      cap en percentil 99%: 1.0312

Después del capping:
       payment_to_income  loan_to_income  revol_util_amount  \
count      391110.000000   391110.000000      391018.000000   
mean            0.079214        0.218107           0.528907   
std             0.041369        0.113035           0.240863   
min             0.000123        0.000309           0.000000   
25%             0.047531        0.129438           0.348536   
50%             0.073038        0.201149           0.531292   
75%             0.105292        0.296610           0.714623   
max             0.193008        0.500000           0.988254   

       total_debt_to_credit  
count         391160.000000  
mean               0.447386  
std                0.277539  
min                0.000000  
25%                0.191137  
50%                0.426327  
75%                0.685724  
max 

## 5. Feature Engineering — Features Temporales

Extraemos información útil de las columnas de fecha.

In [5]:
# Año y mes de emisión del préstamo
df['issue_year'] = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month

# Convertir earliest_cr_line a datetime
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# Antigüedad del historial crediticio en meses al momento del préstamo
df['credit_history_months'] = (
    (df['issue_d'].dt.year - df['earliest_cr_line'].dt.year) * 12 +
    (df['issue_d'].dt.month - df['earliest_cr_line'].dt.month)
)

# Eliminar columnas de fecha originales — ya extrajimos la información útil
df = df.drop(columns=['issue_d', 'earliest_cr_line'])

print("Features temporales creadas:")
print(df[['issue_year', 'issue_month', 'credit_history_months']].describe())

Features temporales creadas:
          issue_year    issue_month  credit_history_months
count  391164.000000  391164.000000          391164.000000
mean     2015.119789       6.727646             199.575319
std         0.587382       3.514194              91.392225
min      2015.000000       1.000000              37.000000
25%      2015.000000       4.000000             137.000000
50%      2015.000000       7.000000             181.000000
75%      2015.000000      10.000000             246.000000
max      2018.000000      12.000000             852.000000


## 6. Feature Engineering — Variables Categóricas

Identificamos la estrategia de encoding para cada variable categórica
según su cardinalidad y naturaleza.

In [6]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f"Columnas categóricas restantes: {len(cat_cols)}")
print()
for col in cat_cols:
    n_unique = df[col].nunique()
    top_value = df[col].value_counts().index[0]
    top_pct = df[col].value_counts(normalize=True).iloc[0] * 100
    print(f"{col:<25} únicos: {n_unique:<6} top: {top_value:<20} ({top_pct:.1f}%)")

Columnas categóricas restantes: 12

term                      únicos: 2      top:  36 months           (75.3%)
grade                     únicos: 7      top: B                    (28.4%)
sub_grade                 únicos: 35     top: C1                   (6.2%)
emp_length                únicos: 11     top: 10+ years            (35.1%)
home_ownership            únicos: 4      top: MORTGAGE             (48.5%)
verification_status       únicos: 3      top: Source Verified      (43.4%)
purpose                   únicos: 14     top: debt_consolidation   (58.8%)
addr_state                únicos: 50     top: CA                   (14.2%)
initial_list_status       únicos: 2      top: w                    (62.7%)
application_type          únicos: 2      top: Individual           (99.3%)
disbursement_method       únicos: 2      top: Cash                 (99.8%)
debt_settlement_flag      únicos: 2      top: N                    (97.0%)


## 6.1 Estrategia de encoding por columna

| Columna | Estrategia | Razón |
|---|---|---|
| `term` | Binary (0/1) | Solo 2 valores: 36 y 60 meses |
| `grade` | Ordinal | Orden natural: A < B < C < D < E < F < G |
| `sub_grade` | Ordinal | Orden natural: A1 < A2 ... < G5 |
| `emp_length` | Ordinal | Orden natural: < 1 year ... 10+ years |
| `home_ownership` | Target Encoding | 4 categorías sin orden claro |
| `verification_status` | Target Encoding | 3 categorías sin orden claro |
| `purpose` | Target Encoding | 14 categorías sin orden claro |
| `addr_state` | Target Encoding | 50 categorías — alta cardinalidad |
| `initial_list_status` | Binary (0/1) | Solo 2 valores: w y f |
| `application_type` | Binary (0/1) | 99.3% Individual — casi constante |
| `disbursement_method` | Binary (0/1) | 99.8% Cash — casi constante |
| `debt_settlement_flag` | Binary (0/1) | Solo 2 valores: N e Y |

In [7]:
for col in cat_cols:
    print(f"Estos son los valores unicos de la columna '{col}': {df[col].unique()}")

Estos son los valores unicos de la columna 'term': [' 36 months' ' 60 months']
Estos son los valores unicos de la columna 'grade': ['C' 'B' 'F' 'A' 'E' 'D' 'G']
Estos son los valores unicos de la columna 'sub_grade': ['C4' 'C1' 'B4' 'F1' 'C3' 'B2' 'B1' 'A2' 'B5' 'C2' 'E2' 'A4' 'E3' 'C5'
 'A1' 'D4' 'F3' 'D1' 'B3' 'D3' 'D5' 'A5' 'F2' 'E4' 'D2' 'E1' 'F5' 'E5'
 'A3' 'G2' 'G1' 'G3' 'G4' 'F4' 'G5']
Estos son los valores unicos de la columna 'emp_length': ['10+ years' '3 years' '4 years' '6 years' '7 years' '8 years' '2 years'
 '5 years' '9 years' '< 1 year' '1 year' None]
Estos son los valores unicos de la columna 'home_ownership': ['MORTGAGE' 'RENT' 'OWN' 'ANY']
Estos son los valores unicos de la columna 'verification_status': ['Not Verified' 'Source Verified' 'Verified']
Estos son los valores unicos de la columna 'purpose': ['debt_consolidation' 'small_business' 'home_improvement' 'major_purchase'
 'credit_card' 'other' 'house' 'vacation' 'car' 'medical' 'moving'
 'renewable_energy' 'weddi

In [8]:
# term: 36 meses = 0, 60 meses = 1
df['term'] = df['term'].str.strip().map({'36 months': 0, '60 months': 1})

# initial_list_status: w = 0, f = 1
df['initial_list_status'] = df['initial_list_status'].map({'w': 0, 'f': 1})

# application_type: Individual = 0, Joint App = 1
df['application_type'] = df['application_type'].map({'Individual': 0, 'Joint App': 1})

# disbursement_method: Cash = 0, DirectPay = 1
df['disbursement_method'] = df['disbursement_method'].map({'Cash': 0, 'DirectPay': 1})

# debt_settlement_flag: N = 0, Y = 1
df['debt_settlement_flag'] = df['debt_settlement_flag'].map({'N': 0, 'Y': 1})

print("Encoding binario completado:")
print(df[['term', 'initial_list_status', 'application_type',
          'disbursement_method', 'debt_settlement_flag']].value_counts().head(10))

Encoding binario completado:
term  initial_list_status  application_type  disbursement_method  debt_settlement_flag
0     0                    0                 0                    0                       169959
      1                    0                 0                    0                       115816
1     0                    0                 0                    0                        66029
      1                    0                 0                    0                        24496
      0                    0                 0                    1                         3446
0     0                    0                 0                    1                         3228
      1                    0                 0                    1                         3165
1     1                    0                 0                    1                         1707
0     0                    1                 0                    0                         1101
1     0    

## 6.2 Encoding ordinal

Variables con orden natural — respetamos ese orden asignando números progresivos.

In [9]:
# grade: A=0, B=1, C=2, D=3, E=4, F=5, G=6
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
df['grade'] = df['grade'].map({g: i for i, g in enumerate(grade_order)})

# sub_grade: A1=0, A2=1, ... G5=34
sub_grades = [f'{g}{n}' for g in 'ABCDEFG' for n in range(1, 6)]
df['sub_grade'] = df['sub_grade'].map({s: i for i, s in enumerate(sub_grades)})

# emp_length: orden natural de experiencia laboral
emp_length_order = {
    '< 1 year': 0,
    '1 year':   1,
    '2 years':  2,
    '3 years':  3,
    '4 years':  4,
    '5 years':  5,
    '6 years':  6,
    '7 years':  7,
    '8 years':  8,
    '9 years':  9,
    '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_length_order)

print("Encoding ordinal completado:")
print(f"grade     — únicos: {df['grade'].nunique()} — valores: {sorted(df['grade'].unique())}")
print(f"sub_grade — únicos: {df['sub_grade'].nunique()} — valores: {sorted(df['sub_grade'].dropna().unique())}")
print(f"emp_length — únicos: {df['emp_length'].nunique()} — valores: {sorted(df['emp_length'].dropna().unique())}")

Encoding ordinal completado:
grade     — únicos: 7 — valores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
sub_grade — únicos: 35 — valores: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34)]
emp_length — únicos: 11 — valores: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0)]


## 6.3 Definición de valores válidos por columna

Documentamos los valores válidos para cada columna categórica.
Esto sirve como referencia para la validación en la API de FastAPI
y para detectar valores inesperados en producción.

In [10]:
# Valores válidos para columnas ordinales
valores_validos_ordinales = {
    'grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'sub_grade': [f'{g}{n}' for g in 'ABCDEFG' for n in range(1, 6)],
    'emp_length': ['< 1 year', '1 year', '2 years', '3 years', '4 years',
                   '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years']
}

# Valores válidos para columnas categóricas (irán a TargetEncoder)
valores_validos_categoricos = {
    'home_ownership':      df['home_ownership'].unique().tolist(),
    'verification_status': df['verification_status'].unique().tolist(),
    'purpose':             df['purpose'].unique().tolist(),
    'addr_state':          df['addr_state'].unique().tolist()
}

# Valores más frecuentes por columna — fallback para valores desconocidos en producción
valores_frecuentes = {
    col: df[col].value_counts().index[0]
    for col in list(valores_validos_categoricos.keys()) +
               ['term', 'initial_list_status', 'application_type',
                'disbursement_method', 'debt_settlement_flag']
}

print("Valores válidos — columnas ordinales:")
for col, vals in valores_validos_ordinales.items():
    print(f"  {col:<12} — {len(vals)} valores")

print("\nValores válidos — columnas categóricas:")
for col, vals in valores_validos_categoricos.items():
    print(f"  {col:<25} — {len(vals)} valores: {vals}")

print("\nValores más frecuentes (fallback en producción):")
for col, val in valores_frecuentes.items():
    print(f"  {col:<30} — '{val}'")

Valores válidos — columnas ordinales:
  grade        — 7 valores
  sub_grade    — 35 valores
  emp_length   — 11 valores

Valores válidos — columnas categóricas:
  home_ownership            — 4 valores: ['MORTGAGE', 'RENT', 'OWN', 'ANY']
  verification_status       — 3 valores: ['Not Verified', 'Source Verified', 'Verified']
  purpose                   — 14 valores: ['debt_consolidation', 'small_business', 'home_improvement', 'major_purchase', 'credit_card', 'other', 'house', 'vacation', 'car', 'medical', 'moving', 'renewable_energy', 'wedding', 'educational']
  addr_state                — 50 valores: ['PA', 'SD', 'IL', 'GA', 'MN', 'SC', 'RI', 'NC', 'CA', 'VA', 'AZ', 'IN', 'MD', 'NY', 'TX', 'KS', 'NM', 'AL', 'WA', 'OH', 'LA', 'FL', 'CO', 'MI', 'MO', 'DC', 'MA', 'WI', 'HI', 'VT', 'NJ', 'DE', 'TN', 'NH', 'NE', 'OR', 'CT', 'AR', 'NV', 'WV', 'MT', 'WY', 'OK', 'KY', 'MS', 'UT', 'ND', 'ME', 'AK', 'ID']

Valores más frecuentes (fallback en producción):
  home_ownership                 — 'MORT

## 6.4 Guardar valores válidos para uso en producción

In [18]:
import json

# Convertir tipos numpy a tipos nativos de Python para serialización JSON
def convertir_a_serializable(obj):
    if isinstance(obj, dict):
        return {k: convertir_a_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convertir_a_serializable(i) for i in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    return obj

valores_produccion = convertir_a_serializable({
    'ordinales': valores_validos_ordinales,
    'categoricos': valores_validos_categoricos,
    'fallbacks': valores_frecuentes
})

MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(MODELS_DIR / 'valores_validos.json', 'w') as f:
    json.dump(valores_produccion, f, indent=2, ensure_ascii=False)

print("Valores válidos guardados en models/valores_validos.json")

Valores válidos guardados en models/valores_validos.json


## 7. Verificación del estado actual del dataset

In [11]:
cat_cols_remaining = df.select_dtypes(include='object').columns.tolist()
print(f"Columnas string restantes ({len(cat_cols_remaining)}):")
print(cat_cols_remaining)
print()
print(f"Shape actual: {df.shape}")

Columnas string restantes (4):
['home_ownership', 'verification_status', 'purpose', 'addr_state']

Shape actual: (391164, 75)


## 8. Imputación de valores nulos

Tratamos los valores nulos restantes en las columnas numéricas.
Usamos mediana para evitar el impacto de outliers.

In [12]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

missing_df = missing_df[missing_df['missing_count'] > 0]

print(f"Columnas con nulos restantes: {len(missing_df)}")
print()
print(missing_df)

Columnas con nulos restantes: 19

                        missing_count  missing_pct
mths_since_last_delinq         190038        48.58
mths_since_recent_inq           40061        10.24
emp_length                      23471         6.00
num_tl_120dpd_2m                18138         4.64
mo_sin_old_il_acct              11922         3.05
percent_bc_gt_75                 4136         1.06
bc_util                          4116         1.05
bc_open_to_buy                   3866         0.99
mths_since_recent_bc             3695         0.94
revol_util_amount                 146         0.04
revol_util                        174         0.04
dti                                57         0.01
payment_to_income                  54         0.01
loan_to_income                     54         0.01
avg_cur_bal                         1         0.00
total_rev_hi_lim                    1         0.00
num_rev_accts                       1         0.00
tot_hi_cred_lim                     3         0.

## 8.1 Features binarias para nulos informativos

Algunos nulos contienen información valiosa — el cliente nunca tuvo 
delinquencias o consultas recientes. Capturamos esa información antes de imputar.

In [13]:
# 1 = nunca tuvo delinquencia, 0 = sí tuvo delinquencia
df['no_delinquency'] = df['mths_since_last_delinq'].isnull().astype(int)

# 1 = sin consultas recientes al buró, 0 = tiene consultas recientes
df['no_recent_inq'] = df['mths_since_recent_inq'].isnull().astype(int)

print("Features binarias creadas:")
print(f"no_delinquency — distribución:")
print(df['no_delinquency'].value_counts())
print()
print(f"no_recent_inq — distribución:")
print(df['no_recent_inq'].value_counts())

Features binarias creadas:
no_delinquency — distribución:
no_delinquency
0    201126
1    190038
Name: count, dtype: int64

no_recent_inq — distribución:
no_recent_inq
0    351103
1     40061
Name: count, dtype: int64


## 8.2 Imputación con mediana

Imputamos los valores nulos restantes con la mediana de cada columna.
Usamos mediana en lugar de media porque es robusta a outliers.

In [14]:
num_cols_with_nulls = df.select_dtypes(include='number').columns[
    df.select_dtypes(include='number').isnull().any()
].tolist()

# Removemos el target
num_cols_with_nulls = [col for col in num_cols_with_nulls if col != 'target']

print(f"Columnas a imputar: {len(num_cols_with_nulls)}")

for col in num_cols_with_nulls:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"  {col:<30} imputado con mediana: {median_val:.4f}")

print(f"\nNulos restantes: {df.isnull().sum().sum()}")

Columnas a imputar: 19
  emp_length                     imputado con mediana: 6.0000
  dti                            imputado con mediana: 18.3200
  mths_since_last_delinq         imputado con mediana: 31.0000
  revol_util                     imputado con mediana: 52.8000
  total_rev_hi_lim               imputado con mediana: 24100.0000
  avg_cur_bal                    imputado con mediana: 7010.0000
  bc_open_to_buy                 imputado con mediana: 4500.0000
  bc_util                        imputado con mediana: 64.1000
  mo_sin_old_il_acct             imputado con mediana: 129.0000
  mths_since_recent_bc           imputado con mediana: 13.0000
  mths_since_recent_inq          imputado con mediana: 5.0000
  num_rev_accts                  imputado con mediana: 13.0000
  num_tl_120dpd_2m               imputado con mediana: 0.0000
  percent_bc_gt_75               imputado con mediana: 50.0000
  tot_hi_cred_lim                imputado con mediana: 107398.0000
  payment_to_income    

## 9. Verificación final del dataset

In [15]:
print(f"Shape final: {df.shape}")
print(f"Default rate: {df['target'].mean():.2%}")
print(f"Nulos: {df.isnull().sum().sum()}")
print()
print("Tipos de datos:")
print(df.dtypes.value_counts())
print()
print("Columnas string (irán a target encoding en modelado):")
print(df.select_dtypes(include='object').columns.tolist())

Shape final: (391164, 77)
Default rate: 20.15%
Nulos: 0

Tipos de datos:
float64    60
int64      10
object      4
int32       3
Name: count, dtype: int64

Columnas string (irán a target encoding en modelado):
['home_ownership', 'verification_status', 'purpose', 'addr_state']


## 10. Guardar dataset de features

In [16]:
df.to_parquet(DATA_PROCESSED / 'loan_features.parquet', index=False)

print(f"Dataset guardado en data/processed/loan_features.parquet")
print(f"Shape: {df.shape}")

Dataset guardado en data/processed/loan_features.parquet
Shape: (391164, 77)
